# Python for AI — Class 18 (Extra)
### How do KNN and the Decision Tree guess a house price?

In Step 3 we tried 3 models. Linear Regression learns a **line** (y = β0 + β1 · x).

**KNN** and the **Decision Tree** do **not** learn a line. Today we open them and see what they do inside, with the same `houses.csv`.

### New words today

| Word | Meaning |
|------|---------|
| **neighbour** | a house in the training data that is **similar** to our new house |
| **distance** | a number that says how **different** two houses are (small = very similar) |
| **K** | how many neighbours KNN looks at (we use 5) |
| **split** | one yes/no question in the tree, like *"size ≤ 10.5?"* |
| **leaf** | the end of a branch. It gives the answer |
| **max_depth** | how many questions the tree can ask, one after another |

## Step 1: Load the data and split (same as class)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

houses = pd.read_csv("houses.csv")
features = ["size_marla", "bedrooms", "age_years", "distance_km"]

X = houses[features]
y = houses["price_lakh"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", len(X_train), "houses")
print("Test: ", len(X_test), "houses")

We follow **one new house** from the test data. The model has **never seen** it.

In [ ]:
new_house = X_test.iloc[[0]]      # first house in the test data
real_price = y_test.iloc[0]

print(new_house)
print("Real price:", real_price, "lakh")

# Part 1: KNN (K-Nearest Neighbours)

- **Idea:** similar houses have similar prices.
- `fit` does **no maths**. It only **saves** the 160 training houses.
- `predict` finds the **5 most similar** houses and takes their **average price**.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)      # only saves the training houses

## Step 2: Find the 5 nearest houses

`kneighbors` tells us **which** 5 houses KNN picked and **how far** each one is.

In [ ]:
distances, rows = knn.kneighbors(new_house)

neighbours = X_train.iloc[rows[0]].copy()
neighbours["price_lakh"] = y_train.iloc[rows[0]].values
neighbours["distance"] = distances[0].round(2)
neighbours

- All 5 houses are **15 to 18 marla** with **6 or 7 bedrooms**, like our new house (16 marla, 7 bedrooms).
- The closest one (distance **2.29**) is almost the same house.

## Step 3: How is the distance calculated?

For each column: **subtract**, then **square**. Add them all up. Take the **square root**.

Example: new house vs the closest neighbour:

| | size | bedrooms | age | km |
|--|--|--|--|--|
| new house | 16 | 7 | 3 | 8.2 |
| neighbour | 17 | 7 | 1 | 7.7 |
| difference² | 1 | 0 | 4 | 0.25 |

√(1 + 0 + 4 + 0.25) = √5.25 = **2.29**

In [ ]:
import numpy as np

a = new_house.iloc[0].values      # the new house
b = neighbours.iloc[0][features].values.astype(float)   # closest neighbour

distance = np.sqrt(((a - b) ** 2).sum())
print("Distance by hand:", round(distance, 2))

## Step 4: Take the average price

In [ ]:
print("Average of the 5 prices:", round(neighbours["price_lakh"].mean(), 2))
print("knn.predict:            ", knn.predict(new_house).round(2))
print("Real price:             ", real_price)

Same number. **KNN = the average price of the 5 most similar houses.**

### Quick Check 1
Use **K = 3**. Take the average of the **first 3** neighbours by hand. Then check with `KNeighborsRegressor(n_neighbors=3)`.

In [ ]:
# your code here

**Answer:**

In [ ]:
print("By hand:", round(neighbours["price_lakh"].head(3).mean(), 2))

knn3 = KNeighborsRegressor(n_neighbors=3)
knn3.fit(X_train, y_train)
print("Model:  ", knn3.predict(new_house).round(2))

# Part 2: Decision Tree

- **Idea:** ask **yes/no questions**, like a flowchart.
- `fit` **learns the questions**. It tries many questions and keeps the one that puts houses with **similar prices** in the same group.
- Each **leaf** (end of a branch) stores the **average price** of the training houses that end there.
- `predict` sends the new house down the tree and gives that leaf's price.

We use `max_depth=2` (only 2 questions), so the tree is small enough to read.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

tree = DecisionTreeRegressor(max_depth=2, random_state=42)
tree.fit(X_train, y_train)      # learns the questions

print(export_text(tree, feature_names=features))

The tree only asks about **size**. Size is the strongest feature (remember the correlation in Step 1 of class).

Draw it as a picture:

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(12, 5))
plot_tree(tree, feature_names=features, filled=True, rounded=True)
plt.show()

- `samples` = how many training houses reached that box.
- `value` = their **average price**.

## Step 5: Check one leaf by hand

The leaf **size > 15.5** says **182.99**. That should be the average price of all training houses bigger than 15.5 marla.

In [ ]:
big = X_train["size_marla"] > 15.5
print("Houses in this leaf:", big.sum())
print("Their average price:", round(y_train[big].mean(), 2))

## Step 6: Follow our new house down the tree

New house = **16 marla**.
1. size ≤ 10.5? **No** → go right
2. size ≤ 15.5? **No** → go right
3. Leaf: **182.99 lakh**

In [ ]:
print("tree.predict:", tree.predict(new_house).round(2))
print("Real price:  ", real_price)

### Quick Check 2
A house is **8 marla**, 4 bedrooms, 10 years old, 12 km away. Follow the tree **by hand** first. Then check with `tree.predict`.

In [ ]:
# your code here

**Answer:** 8 ≤ 10.5 → **Yes**. 8 ≤ 5.5 → **No**. Leaf = **80.09 lakh**.

In [ ]:
house_8 = pd.DataFrame({"size_marla": [8], "bedrooms": [4], "age_years": [10], "distance_km": [12]})
print(tree.predict(house_8).round(2))

# Part 3: All 3 models on the same house

In [ ]:
from sklearn.linear_model import LinearRegression

line = LinearRegression()
line.fit(X_train, y_train)

print("Real price:        ", real_price)
print("Linear Regression: ", line.predict(new_house).round(1))
print("KNN (K=5):         ", knn.predict(new_house).round(1))
print("Decision Tree:     ", tree.predict(new_house).round(1))

| Model | What `fit` does | What `predict` does |
|-------|-----------------|---------------------|
| **Linear Regression** | finds β0 and β1 (the best line) | puts the numbers into the line |
| **KNN** | only **saves** the training houses | average price of the **5 nearest** houses |
| **Decision Tree** | learns the **yes/no questions** | follows the questions, gives the leaf's average |

Only Linear Regression is an **equation**. KNN **remembers**. The tree **asks questions**.

All 3 models give a good guess for **this one house**. To choose the best model, we still check the error on **all** the test houses (like Step 3 of class).

## The complete program

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, export_text

# Step 1: load and split
houses = pd.read_csv("houses.csv")
features = ["size_marla", "bedrooms", "age_years", "distance_km"]
X = houses[features]
y = houses["price_lakh"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
new_house = X_test.iloc[[0]]

# KNN: find the 5 nearest houses, take their average price
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)
distances, rows = knn.kneighbors(new_house)
print("5 nearest prices:", y_train.iloc[rows[0]].values)
print("KNN guess:", knn.predict(new_house).round(1))

# Decision Tree: learn the questions, follow them
tree = DecisionTreeRegressor(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=features))
print("Tree guess:", tree.predict(new_house).round(1))
print("Real price:", y_test.iloc[0])

# Home Problem: cars.csv

Do the same steps with `cars.csv`. Features: `car_age`, `km_thousands`, `engine_cc`. Target: `price_lakh`.

1. Load the data. Split with `test_size=0.2, random_state=42`.
2. Take the first test car as `new_car`. Print it and its real price.
3. **KNN:** fit with K = 5. Show the 5 nearest cars with their prices and distances.
4. Take the average of the 5 prices by hand. Check it with `knn.predict`.
5. **Tree:** fit with `max_depth=2`. Print it with `export_text`.
6. Follow `new_car` down the tree by hand. Check with `tree.predict`.
7. **Think:** look at the 5 nearest cars. Which column decides the distance the most? Why? (Hint: look at how big the numbers in each column are.)

In [ ]:
# your code here